# Chapter 10 — Dynamic Programming

*Built with [Claude](https://claude.ai) by Anthropic.*

Dynamic programming solves problems with overlapping subproblems by storing results so they're never recomputed. The Viterbi algorithm, DTW, CTC loss — these are all DP. You've been using it without calling it that.

In [1]:
from functools import lru_cache
from typing import List

def show_dp(dp, label='dp', highlight=None):
    """Print a 1D DP table."""
    vals = '  '.join(f"{v:>4}" for v in dp)
    idxs = '  '.join(f"{i:>4}" for i in range(len(dp)))
    print(f"  {label} vals: {vals}")
    print(f"  {label} idx:  {idxs}")

def show_dp2d(dp, row_label='i', col_label='j'):
    """Print a 2D DP table."""
    cols = len(dp[0])
    header = '     ' + '  '.join(f"{j:>4}" for j in range(cols))
    print(f"  {header}")
    for i, row in enumerate(dp):
        print(f"  {i:>3}: " + '  '.join(f"{v:>4}" for v in row))

print('  Helpers loaded: show_dp, show_dp2d')

  Helpers loaded: show_dp, show_dp2d


---

## Part 1 — 1D DP (Linear Sequence)

**DS/ML connection:** Markov chain state transitions, sequence models — `dp[i]` is the optimal value at position `i`, updated from a fixed number of previous positions. This is the same recurrence as `cumsum` but with a `max` or `min` instead of `+`.

In [2]:
# ── DS/ML PARALLEL: cumulative max (pandas) ──
import pandas as pd
import numpy as np

# Fibonacci via cumsum analogy
n = 8
fib = [0, 1]
for _ in range(2, n):
    fib.append(fib[-1] + fib[-2])
print(f"  Fibonacci up to n={n}: {fib}")
print(f"  (each value = sum of previous 2 — same structure as dp[i] = dp[i-1] + dp[i-2])")

  Fibonacci up to n=8: [0, 1, 1, 2, 3, 5, 8, 13]
  (each value = sum of previous 2 — same structure as dp[i] = dp[i-1] + dp[i-2])


### LC 70 — Climbing Stairs

**Problem:** You can climb 1 or 2 steps at a time. How many distinct ways are there to reach step `n`?

**DS parallel:** Counting paths in a 2-step Markov chain — the recurrence is identical to computing transition counts.

**Key insight:** `dp[i] = dp[i-1] + dp[i-2]` — Fibonacci recurrence. Space-optimize to two variables.

In [3]:
# ── DS/ML APPROACH: precompute full table ──
def climbStairs_table(n):
    if n <= 2: return n
    dp = [0] * (n + 1)
    dp[1], dp[2] = 1, 2
    for i in range(3, n + 1):
        dp[i] = dp[i-1] + dp[i-2]
    return dp

# ── RAW PYTHON (interview answer) ──
def climbStairs(n: int) -> int:
    if n <= 2: return n
    a, b = 1, 2
    for _ in range(3, n + 1):
        a, b = b, a + b
    return b

n = 8
table = climbStairs_table(n)
print(f"  n={n}")
show_dp(table, 'dp')
print(f"  climbStairs({n}) = {climbStairs(n)}")

  n=8
  dp vals:    0     1     2     3     5     8    13    21    34
  dp idx:     0     1     2     3     4     5     6     7     8
  climbStairs(8) = 34


In [4]:
# ── TRACE: LC 70 ──
def climbStairs_trace(n):
    if n <= 2:
        print(f"  n={n} ≤ 2 → return {n}")
        return n
    a, b = 1, 2
    print(f"  n={n}")
    print(f"  {'step':>5}  {'a (i-1)':>9}  {'b (i)':>7}  {'new b = a+b':>12}")
    print(f"  {'2':>5}  {'1':>9}  {'2':>7}  {'(base)':>12}")
    for i in range(3, n + 1):
        new_b = a + b
        print(f"  {i:>5}  {a:>9}  {b:>7}  {new_b:>12}")
        a, b = b, new_b
    print(f"  ways to climb {n} stairs = {b}")

climbStairs_trace(8)

  n=8
   step    a (i-1)    b (i)   new b = a+b
      2          1        2        (base)
      3          1        2             3
      4          2        3             5
      5          3        5             8
      6          5        8            13
      7          8       13            21
      8         13       21            34
  ways to climb 8 stairs = 34


### LC 198 — House Robber

**Problem:** Given house values, maximize the total robbed without taking two adjacent houses.

**DS parallel:** Optimal non-adjacent feature selection — pick a subset of time series features such that no two selected features are adjacent in the sequence.

**Key insight:** `dp[i] = max(dp[i-1], dp[i-2] + nums[i])` — at each house, either skip it (take `dp[i-1]`) or rob it (take `dp[i-2] + nums[i]`).

In [5]:
# ── DS/ML APPROACH: full DP table ──
def rob_table(nums):
    if not nums: return 0
    if len(nums) == 1: return nums[0]
    dp = [0] * len(nums)
    dp[0] = nums[0]
    dp[1] = max(nums[0], nums[1])
    for i in range(2, len(nums)):
        dp[i] = max(dp[i-1], dp[i-2] + nums[i])
    return dp

# ── RAW PYTHON (interview answer) ──
def rob(nums: List[int]) -> int:
    prev2, prev1 = 0, 0
    for num in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + num)
    return prev1

nums = [2, 7, 9, 3, 1]
table = rob_table(nums)
print(f"  nums: {nums}")
show_dp(table, 'dp')
print(f"  max rob = {rob(nums)}")

  nums: [2, 7, 9, 3, 1]
  dp vals:    2     7    11    11    12
  dp idx:     0     1     2     3     4
  max rob = 12


In [6]:
# ── TRACE: LC 198 ──
def rob_trace(nums):
    prev2, prev1 = 0, 0
    print(f"  nums={nums}")
    print(f"  {'i':>3}  {'num':>5}  {'prev2':>7}  {'prev1':>7}  {'max(prev1, prev2+num)':>22}")
    for i, num in enumerate(nums):
        new = max(prev1, prev2 + num)
        print(f"  {i:>3}  {num:>5}  {prev2:>7}  {prev1:>7}  {new:>22}")
        prev2, prev1 = prev1, new
    print(f"  max rob = {prev1}")

rob_trace([2, 7, 9, 3, 1])

  nums=[2, 7, 9, 3, 1]
    i    num    prev2    prev1   max(prev1, prev2+num)
    0      2        0        0                       2
    1      7        0        2                       7
    2      9        2        7                      11
    3      3        7       11                      11
    4      1       11       11                      12
  max rob = 12


---

## Part 2 — Unbounded Knapsack (Coin Change)

**DS/ML connection:** Minimum number of model updates to reach a target loss — if each update reduces loss by a fixed amount, this is coin change. Also maps to minimum number of batch iterations to process a dataset.

### LC 322 — Coin Change

**Problem:** Given `coins` and an `amount`, return the minimum number of coins to make up the amount, or `-1` if impossible.

**DS parallel:** Minimum number of fixed-size model updates to reach a target — each coin denomination is an update size.

**Key insight:** `dp[a] = min(dp[a-c]+1 for c in coins if c<=a)`. Initialize `dp[0]=0`, all others `inf`. Fill left to right.

In [7]:
# ── DS/ML APPROACH: memoized recursion ──
def coinChange_memo(coins, amount):
    @lru_cache(maxsize=None)
    def dp(a):
        if a == 0: return 0
        if a < 0:  return float('inf')
        return min(dp(a - c) + 1 for c in coins)
    result = dp(amount)
    return result if result != float('inf') else -1

# ── RAW PYTHON (interview answer) ──
def coinChange(coins: List[int], amount: int) -> int:
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    for a in range(1, amount + 1):
        for coin in coins:
            if coin <= a:
                dp[a] = min(dp[a], dp[a - coin] + 1)
    return dp[amount] if dp[amount] != float('inf') else -1

tests = [([1,5,10,25], 36), ([2], 3)]
for coins, amount in tests:
    print(f"  coins={coins}, amount={amount}")
    print(f"    memo: {coinChange_memo(coins, amount)}")
    print(f"    dp:   {coinChange(coins, amount)}")

  coins=[1, 5, 10, 25], amount=36
    memo: 3
    dp:   3
  coins=[2], amount=3
    memo: -1
    dp:   -1


In [8]:
# ── TRACE: LC 322 ──
def coinChange_trace(coins, amount):
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    print(f"  coins={coins}, amount={amount}")
    print(f"  Building dp table:")
    for a in range(1, amount + 1):
        for coin in coins:
            if coin <= a and dp[a - coin] + 1 < dp[a]:
                dp[a] = dp[a - coin] + 1
        inf_str = [('∞' if v == float('inf') else str(v)) for v in dp]
    show_dp([('∞' if v == float('inf') else v) for v in dp], 'dp')
    result = dp[amount] if dp[amount] != float('inf') else -1
    print(f"  min coins for {amount} = {result}")

coinChange_trace([1, 5, 10, 25], 36)

  coins=[1, 5, 10, 25], amount=36
  Building dp table:
  dp vals:    0     1     2     3     4     1     2     3     4     5     1     2     3     4     5     2     3     4     5     6     2     3     4     5     6     1     2     3     4     5     2     3     4     5     6     2     3
  dp idx:     0     1     2     3     4     5     6     7     8     9    10    11    12    13    14    15    16    17    18    19    20    21    22    23    24    25    26    27    28    29    30    31    32    33    34    35    36
  min coins for 36 = 3


### LC 139 — Word Break

**Problem:** Given a string `s` and a word dictionary, return `True` if `s` can be segmented into dictionary words.

**DS parallel:** Sequence labeling feasibility — can a text string be covered by known entity spans?

**Key insight:** `dp[i] = True` if `s[:i]` can be segmented. For each `i`, check all `j < i` where `dp[j]` is True and `s[j:i]` is in the word set.

In [9]:
# ── DS/ML APPROACH: memoized recursion ──
def wordBreak_memo(s, wordDict):
    word_set = set(wordDict)
    @lru_cache(maxsize=None)
    def dp(i):
        if i == len(s): return True
        return any(s[i:j] in word_set and dp(j) for j in range(i+1, len(s)+1))
    return dp(0)

# ── RAW PYTHON (interview answer) ──
def wordBreak(s: str, wordDict: List[str]) -> bool:
    word_set = set(wordDict)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True   # empty string is always segmentable
    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                break
    return dp[n]

tests = [
    ('leetcode', ['leet','code']),
    ('applepenapple', ['apple','pen']),
    ('catsandog', ['cats','dog','sand','and','cat'])
]
for s, d in tests:
    print(f"  s={s!r}, dict={d}")
    print(f"    memo: {wordBreak_memo(s, d)}  dp: {wordBreak(s, d)}")

  s='leetcode', dict=['leet', 'code']
    memo: True  dp: True
  s='applepenapple', dict=['apple', 'pen']
    memo: True  dp: True
  s='catsandog', dict=['cats', 'dog', 'sand', 'and', 'cat']
    memo: False  dp: False


In [10]:
# ── TRACE: LC 139 ──
def wordBreak_trace(s, wordDict):
    word_set = set(wordDict)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True
    print(f"  s={s!r}, dict={wordDict}")
    for i in range(1, n + 1):
        for j in range(i):
            segment = s[j:i]
            if dp[j] and segment in word_set:
                dp[i] = True
                print(f"  dp[{i}]=True  because dp[{j}]=True and {segment!r} in dict")
                break
    dp_str = [str(int(v)) for v in dp]
    print(f"  dp table: {dp_str}")
    print(f"  result: {dp[n]}")

wordBreak_trace('leetcode', ['leet','code'])

  s='leetcode', dict=['leet', 'code']
  dp[4]=True  because dp[0]=True and 'leet' in dict
  dp[8]=True  because dp[4]=True and 'code' in dict
  dp table: ['1', '0', '0', '0', '1', '0', '0', '0', '1']
  result: True


---

## Part 3 — 2D DP (Sequence Alignment)

**DS/ML connection:** The DP table in LCS and Edit Distance is structurally identical to the alignment matrix in DTW (dynamic time warping) and the score matrix in Smith-Waterman sequence alignment. You've been using this in ML — now you're seeing the algorithm underneath.

### LC 1143 — Longest Common Subsequence

**Problem:** Given two strings, return the length of their longest common subsequence.

**DS parallel:** The DP table IS the alignment matrix used in DTW — same filling pattern, same recurrence.

**Key insight:** `dp[i][j] = dp[i-1][j-1] + 1` if characters match, else `max(dp[i-1][j], dp[i][j-1])`.

In [11]:
# ── DS/ML APPROACH: memoized recursion ──
def lcs_memo(s1, s2):
    @lru_cache(maxsize=None)
    def dp(i, j):
        if i == 0 or j == 0: return 0
        if s1[i-1] == s2[j-1]: return dp(i-1, j-1) + 1
        return max(dp(i-1, j), dp(i, j-1))
    return dp(len(s1), len(s2))

# ── RAW PYTHON (interview answer) ──
def longestCommonSubsequence(text1: str, text2: str) -> int:
    m, n = len(text1), len(text2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i-1] == text2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

tests = [('abcde', 'ace'), ('abc', 'abc'), ('abc', 'def')]
for s1, s2 in tests:
    print(f"  ({s1!r}, {s2!r}): memo={lcs_memo(s1, s2)}  dp={longestCommonSubsequence(s1, s2)}")

  ('abcde', 'ace'): memo=3  dp=3
  ('abc', 'abc'): memo=3  dp=3
  ('abc', 'def'): memo=0  dp=0


In [12]:
# ── TRACE: LC 1143 ──
def lcs_trace(text1, text2):
    m, n = len(text1), len(text2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i-1] == text2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    print(f"  text1={text1!r}, text2={text2!r}")
    print(f"  DP table (rows=text1, cols=text2):")
    print(f"       {'    '.join([''] + list(text2))}")
    show_dp2d(dp)
    print(f"  LCS length = {dp[m][n]}")

lcs_trace('abcde', 'ace')

  text1='abcde', text2='ace'
  DP table (rows=text1, cols=text2):
           a    c    e
          0     1     2     3
    0:    0     0     0     0
    1:    0     1     1     1
    2:    0     1     1     1
    3:    0     1     2     2
    4:    0     1     2     2
    5:    0     1     2     3
  LCS length = 3


### LC 72 — Edit Distance

**Problem:** Return the minimum number of operations (insert, delete, replace) to convert `word1` to `word2`.

**DS parallel:** String similarity / fuzzy matching — the same 2D DP structure as LCS, used in spell correction and record linkage.

**Key insight:** Three transitions per cell — insert (`dp[i][j-1]+1`), delete (`dp[i-1][j]+1`), replace (`dp[i-1][j-1] + (0 if chars match else 1)`).

In [13]:
# ── DS/ML APPROACH: python-Levenshtein / jellyfish ──
def edit_distance_manual(s1, s2):
    """Pure Python Levenshtein (no external lib needed)."""
    m, n = len(s1), len(s2)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[j] = prev[j-1]
            else:
                dp[j] = 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n]

# ── RAW PYTHON (interview answer) ──
def minDistance(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i   # delete all of word1
    for j in range(n + 1): dp[0][j] = j   # insert all of word2
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i-1] == word2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j],    # delete
                                   dp[i][j-1],    # insert
                                   dp[i-1][j-1])  # replace
    return dp[m][n]

tests = [('horse', 'ros'), ('intention', 'execution')]
for w1, w2 in tests:
    print(f"  ({w1!r} → {w2!r}): space_opt={edit_distance_manual(w1,w2)}  dp={minDistance(w1,w2)}")

  ('horse' → 'ros'): space_opt=3  dp=3
  ('intention' → 'execution'): space_opt=5  dp=5


In [14]:
# ── TRACE: LC 72 ──
def minDistance_trace(word1, word2):
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i-1] == word2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    print(f"  word1={word1!r}, word2={word2!r}")
    print(f"  DP table (rows=word1 chars, cols=word2 chars):")
    header = '     ' + '  '.join(f"{'_':>3}") + '  ' + '  '.join(f"{c:>3}" for c in word2)
    print(f"  {header}")
    labels = ['_'] + list(word1)
    for i, row in enumerate(dp):
        print(f"  {labels[i]:>3}: " + '  '.join(f"{v:>3}" for v in row))
    print(f"  edit distance = {dp[m][n]}")

minDistance_trace('horse', 'ros')

  word1='horse', word2='ros'
  DP table (rows=word1 chars, cols=word2 chars):
             _    r    o    s
    _:   0    1    2    3
    h:   1    1    2    3
    o:   2    2    1    2
    r:   3    2    2    2
    s:   4    3    3    2
    e:   5    4    4    3
  edit distance = 3
